In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


spark = SparkSession.builder.getOrCreate()
spark.sql("create catalog if not exists sql_master")
spark.sql("USE catalog sql_master")
spark.sql("CREATE SCHEMA IF NOT EXISTS sql_master_pack")
spark.sql("USE SCHEMA sql_master_pack")

In [0]:
spark.sql("USE catalog sql_master")
spark.sql("USE SCHEMA sql_master_pack")

spark.sql("""
CREATE OR REPLACE TABLE departments (
    dept_id STRING, dept_name STRING, location STRING, budget DECIMAL(12, 2)
)""")

spark.sql("""
INSERT INTO departments VALUES
('IT', 'Information Technology', 'Sydney', 1000000),
('Sales', 'Sales & Marketing', 'Melbourne', 800000),
('Account', 'Accounting & Finance', 'Sydney', 500000),
('CEO', 'Executive Office', 'Sydney', 400000),
('HR', 'Human Resources', 'Brisbane', 300000)
""")


In [0]:
spark.sql("USE catalog sql_master")
spark.sql("USE SCHEMA sql_master_pack")

spark.sql("""
CREATE OR REPLACE TABLE employees (
    employee_id INT, first_name STRING, last_name STRING,
    dept_id STRING, manager_id INT, salary DOUBLE, expertise STRING
)""")
spark.sql("""
INSERT INTO employees VALUES
(100, 'John',  'White',   'IT',      103,  120000, 'Senior'),
(101, 'Mary',  'Danner',  'Account', 109,   80000, 'junior'),
(102, 'Ann',   'Lynn',    'Sales',   107,  140000, 'Semisenior'),
(103, 'Peter', 'Oconnor', 'IT',      110,  130000, 'Senior'),
(106, 'Sue',   'Sanchez', 'Sales',   107,  110000, 'Junior'),
(107, 'Marta', 'Doe',     'Sales',   110,  180000, 'Senior'),
(109, 'Ann',   'Danner',  'Account', 110,   90000, 'Senior'),
(110, 'Simon', 'Yang',    'CEO',     NULL, 250000, 'Senior'),
(111, 'Juan',  'Graue',   'Sales',   102,   37000, 'Junior')
""")

# 2. EMPLOYEE_SALARY & STAGING_EMPLOYEES (16-Query Window Cheat Sheet + MERGE)
spark.sql("""
CREATE OR REPLACE TABLE employee_salary (
    EmpID INT, EmpName STRING, Department STRING, Salary DECIMAL(10, 2), JoiningDate DATE
)""")
spark.sql("""
INSERT INTO employee_salary VALUES
(101, 'John',  'IT',      8000,  DATE'2025-01-10'),
(102, 'Mary',  'IT',      9000,  DATE'2025-02-15'),
(103, 'David', 'IT',      7000,  DATE'2025-03-20'),
(104, 'Sam',   'HR',      6000,  DATE'2025-01-05'),
(105, 'Lisa',  'HR',      7500,  DATE'2025-04-01'),
(106, 'Mike',  'Finance', 10000, DATE'2025-02-10'),
(107, 'Emma',  'Finance', 9500,  DATE'2025-03-15')
""")

spark.sql("""
CREATE OR REPLACE TABLE staging_employees (
    EmpID INT, EmpName STRING, Department STRING, Salary DECIMAL(10, 2), JoiningDate DATE
)""")
spark.sql("""
INSERT INTO staging_employees VALUES
(101, 'John',  'IT',      8500,  DATE'2025-01-10'),
(108, 'Priya', 'Finance', 11000, DATE'2025-05-01')
""")

# 3. SALES & DAILY_SALES (Core PySpark/SQL examples + Moving Avg + YoY)
sales = spark.createDataFrame([
    (1, "North", "P1", "2024-01-01", 100.0),
    (2, "North", "P1", "2024-01-02", 150.0),
    (3, "North", "P2", "2024-01-01", 200.0),
    (4, "South", "P1", "2024-01-01", 120.0),
    (5, "South", "P2", "2024-01-02", 180.0),
], ["sale_id", "region", "product_id", "sale_date", "amount"]) \
.withColumn("sale_date", F.to_date("sale_date"))
sales.write.mode("overwrite").saveAsTable("sales")

spark.sql("""
CREATE OR REPLACE TABLE daily_sales (sale_date DATE, daily_amount DECIMAL(10, 2))
""")
spark.sql("""
INSERT INTO daily_sales VALUES
(DATE'2024-01-01', 1000), (DATE'2024-01-02', 1200), (DATE'2024-01-03', 1100),
(DATE'2025-01-01', 1500), (DATE'2025-01-02', 1800), (DATE'2025-01-03', 1400),
(DATE'2025-01-04', 2000), (DATE'2025-01-05', 2200), (DATE'2025-01-06', 1900),
(DATE'2025-01-07', 2500), (DATE'2025-01-08', 2300), (DATE'2025-01-09', 2600),
(DATE'2026-01-01', 3000), (DATE'2026-01-02', 3200), (DATE'2026-01-03', 3100)
""")

# 4. CLOUD_SPEND (FinOps: Variance, Pareto 80/20, Spike Detection, MoM, YoY)
spark.sql("""
CREATE OR REPLACE TABLE cloud_spend (
    spend_month DATE, environment STRING, service STRING, cost_usd DECIMAL(10, 2)
)""")
spark.sql("""
INSERT INTO cloud_spend VALUES
(DATE'2025-01-01', 'Prod', 'Compute',  12000),
(DATE'2025-02-01', 'Prod', 'Compute',  12500),
(DATE'2025-03-01', 'Prod', 'Compute',  28000),
(DATE'2025-04-01', 'Prod', 'Compute',  13000),
(DATE'2025-01-01', 'Prod', 'Database',  8000),
(DATE'2025-02-01', 'Prod', 'Database',  8200),
(DATE'2025-01-01', 'Dev',  'Compute',   2000),
(DATE'2025-02-01', 'Dev',  'Storage',    500),
(DATE'2026-01-01', 'Prod', 'Compute',  15000),
(DATE'2026-02-01', 'Prod', 'Compute',  14000)
""")

# 5. CUSTOMERS, ORDERS, CUSTOMER_RECORDS (Joins, Anti-Joins, Dedup, NULLs, Gaps)
spark.sql("CREATE OR REPLACE TABLE customers (cust_id INT, cust_name STRING, email STRING)")
spark.sql("INSERT INTO customers VALUES (1, 'Alice', 'alice@mail.com'), (2, 'Bob', NULL), (3, 'Cara', 'cara@mail.com')")

spark.sql("""
CREATE OR REPLACE TABLE orders (
    order_id INT, cust_id INT, product STRING, order_dt DATE, status STRING, amount DECIMAL(10, 2)
)""")
spark.sql("""
INSERT INTO orders VALUES
(1001, 1, 'A', DATE'2025-01-05', 'Shipped',   100),
(1002, 1, 'B', DATE'2025-01-10', 'Delivered', 200),
(1004, 1, 'A', DATE'2025-02-01', 'Delivered', 150),
(1005, 2, 'A', DATE'2025-01-07', 'Pending',   300),
(1005, 2, 'A', DATE'2025-01-07', 'Shipped',   300)
""")

# 6. USER_LOGINS, EVENTS (Islands & Sessionization) & BIKE_TRIPS (Frames)
spark.sql("CREATE OR REPLACE TABLE user_logins (user_id INT, login_date DATE)")
spark.sql("""
INSERT INTO user_logins VALUES
(1, DATE'2026-01-01'), (1, DATE'2026-01-01'), -- Intentional duplicate same-day login!
(1, DATE'2026-01-02'), (1, DATE'2026-01-03'), (1, DATE'2026-01-04'),
(1, DATE'2026-01-07'), (1, DATE'2026-01-08'),
(1, DATE'2026-01-12'), (1, DATE'2026-01-13'), (1, DATE'2026-01-14')
""")

spark.sql("CREATE OR REPLACE TABLE events (event_id INT, user_id INT, event_time TIMESTAMP)")
spark.sql("""
INSERT INTO events VALUES
(1, 10, TIMESTAMP'2026-01-01 09:00:00'),
(2, 10, TIMESTAMP'2026-01-01 09:15:00'), -- Same session (15m gap)
(3, 10, TIMESTAMP'2026-01-01 10:00:00'), -- New session (45m > 30m gap)
(4, 20, TIMESTAMP'2026-01-01 09:05:00')
""")

spark.sql("""
CREATE OR REPLACE TABLE bike_trips (
    trip_id INT, bike_number STRING, start_date TIMESTAMP, start_station_id INT, end_station_id INT
)""")
spark.sql("""
INSERT INTO bike_trips VALUES
(1, 'B101', TIMESTAMP'2025-10-25 08:15:00', 10, 22),
(2, 'B101', TIMESTAMP'2025-10-25 11:30:00', 22, 35),
(3, 'B101', TIMESTAMP'2025-10-25 17:45:00', 35, 14),
(4, 'B202', TIMESTAMP'2025-10-25 09:00:00', 50, 61),
(5, 'B202', TIMESTAMP'2025-10-25 14:20:00', 61, 50)
""")

# 7. ARTICLES & ORDERS_COMPLEX (Arrays, Maps, JSON, Lambdas)
spark.createDataFrame(
    [(1, "Python", "python,data,tutorial"), (2, "SQL", None)],
    ["article_id", "title", "tags"]
).write.mode("overwrite").saveAsTable("articles")

spark.createDataFrame(
    [(1, "alice", ["laptop", "mouse"], {"priority": "high"}, '{"ship":"air","days":2}', [10.0, 20.0, 30.0]),
     (2, "bob",   ["keyboard"],        {"priority": "low"},  '{"ship":"sea","days":9}', [5.0, -1.0])],
    "order_id INT, customer STRING, items ARRAY<STRING>, meta MAP<STRING,STRING>, shipping STRING, amounts ARRAY<DOUBLE>"
).write.mode("overwrite").saveAsTable("orders_complex")